# TEJ Wizard 資料庫使用指南

這份指南寫給**會 Python、常用 Pandas，但沒有學過 SQL 或資料庫課程**的使用者。完成後，你會知道：

- 這個資料庫有哪些資料、沒有涵蓋哪些資料
- 如何用 Python 連線，並把查詢結果拿成 `DataFrame`
- 如何查單一股票、多檔股票與指定日期範圍
- SQL 最常用的五個關鍵字，及其 Pandas 對照
- 如何檢查資料新鮮度與缺口

> **先記住三件事：**連線一律使用 `read_only=True`；股票代碼是字串（例如 `'0050'`）；查日期範圍優先使用 `adj_daily_prices_between()`。


## 1. 先用 Pandas 的方式理解資料庫

可以先把這個 DuckDB 資料庫想成一個「替你管理很多 Parquet 檔案的查詢入口」：

| 資料庫概念 | 可類比的 Pandas 概念 | 本專案例子 |
|---|---|---|
| database | 一組相關資料的入口 | `tej_wizard.duckdb` |
| view | 已定義好來源與欄位的唯讀 DataFrame | `v_adj_daily_prices` |
| column | DataFrame 欄位 | `date`, `stock_id`, `close` |
| row | 一筆 observation | 某證券在某交易日的資料 |
| SQL query | 篩選、選欄、排序的操作鏈 | `SELECT ... WHERE ...` |

實際資料存放在依年、月分區的 Parquet 檔；`.duckdb` 檔主要保存 view 定義。一般分析只要查 view，不需要自行打開 raw Excel 或 Parquet。


## 2. 準備環境並連線

需要 `duckdb` 與 `pandas`。若下一格出現 `ModuleNotFoundError`，可在新的 cell 執行：

```python
%pip install duckdb pandas pyarrow
```

安裝後請重新啟動 kernel。下方程式會從目前資料夾一路往上尋找 repo 根目錄，因此從本 notebook 目錄或專案根目錄啟動 Jupyter 都可以。


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd


def find_database() -> Path:
    """從目前目錄往上找 db/tej-wizard/tej_wizard.duckdb。"""
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        candidate = parent / "db" / "tej-wizard" / "tej_wizard.duckdb"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "找不到 db/tej-wizard/tej_wizard.duckdb；"
        "請確認你是在 financial-db repo 內啟動 Jupyter。"
    )


DB_PATH = find_database()
con = duckdb.connect(str(DB_PATH), read_only=True)
pd.set_option("display.max_columns", 50)

print(f"已用唯讀模式連線：{DB_PATH}")


> 為什麼一定要唯讀？DuckDB 同一時間只允許一個 writer。互動式 notebook 若持有可寫連線，可能擋住每日更新工作。分析時不應在這個 DB 建 table、view 或 index。


## 3. 這裡有哪些資料？

這個資料庫有**兩個 dataset**，不要混為一談：

| | `v_adj_daily_prices` | `v_index_constituents` |
|---|---|---|
| 內容 | 個股調整股價（日頻） | 台灣50／台灣中型100 的每日官方成分 |
| 一列 | 一檔證券 × 一個交易日 | 一個指數 × 一個交易日 × 一檔成分股 |
| 鍵 | `(stock_id, date)` | `(index_id, date, stock_id)` |
| 來源 | TEJ `waprcd1` | TEJ `widxs` |

核心資料是 TEJ `waprcd1` 的**個股調整股價**。證券範圍是 **TWN50 ∪ TM100 歷年成分聯集 365 檔，加上 0050、0051 兩支 ETF 本身，共 367 檔**——包含已下市或合併者，這正是這個資料庫的價值所在。兩支 ETF 本身都不是任何指數的成分。下面兩格會從目前的 DB 重新計算實際數字。

> **這不是全市場資料。** 做橫斷面、排名或選股時，結論只適用於這個證券聯集；不要把它描述成整個台股市場。

### 成分清單從哪裡來？

來自 TEJ `widxs` 的 48 份「指數 × 年度」workbook，已經整合成 `v_index_constituents`，**直接用 SQL 查就好，不需要另外讀檔**。它是 point-in-time membership 的權威來源。

`raw/daily-0050-constituents/0050_daily_constituents.feather` 仍然存在，但**已不是 membership 來源**：它記的是 0050 **ETF 自身**的 review snapshot 持股（不是指數成分），而且停在 2026-07-31。只有需要 ISIN 或 ETF 自身權重時才讀它。


In [ ]:
index_sql = """
SELECT
    index_id,
    count(*) AS row_count,
    count(DISTINCT date) AS dates,
    count(DISTINCT stock_id) AS constituent_union,
    min(date) AS first_date,
    max(date) AS last_date
FROM v_index_constituents
GROUP BY index_id
ORDER BY index_id
"""

con.execute(index_sql).df()

In [ ]:
summary_sql = """
SELECT
    count(*) AS rows,
    count(DISTINCT stock_id) AS securities,
    count(DISTINCT date) AS trading_days,
    min(date) AS first_date,
    max(date) AS last_date
FROM v_adj_daily_prices
"""

con.execute(summary_sql).df()


### 想知道「某一天的成分股是哪些」？

用 `index_members_on(index_id, as_of)`。歷史回測**一定要**這樣取當日成分，不能把 365 檔聯集放進每一天——後者會把日後才入選的公司放進過去，形成 look-ahead bias。

這裡刻意用 `LEFT JOIN` 而不是 `INNER JOIN`：先看得到「當日是成分股但當天沒有價格」的列（停牌），不要一開始就把問題靜默丟掉。


In [ ]:
members_sql = """
SELECT m.stock_id, m.stock_name, m.weight_pct, p.close
FROM index_members_on('TWN50', DATE '2020-01-02') m
LEFT JOIN adj_daily_prices_between('2020-01-02', '2020-01-02') p
  ON p.stock_id = m.stock_id
ORDER BY m.weight_pct DESC NULLS LAST
"""

con.execute(members_sql).df().head(10)

### 可查詢的 view

| View | 粒度與用途 |
|---|---|
| `v_adj_daily_prices` | 主資料；每檔證券每交易日一列，共 40 欄 |
| `v_securities` | 每檔證券一列；名稱、市場、首末日期、是否仍有最新資料 |
| `v_trading_calendar` | 資料中曾出現的交易日，每日一列 |
| `v_last_trading_day` | 全庫最新交易日，只有一列 |
| `v_daily_coverage` | 每日有幾列、幾檔證券及最後匯入時間 |
| `v_thin_trading_days` | 相對鄰近日明顯少資料的可疑日期 |
| `v_security_gaps` | 個別證券活躍期間內缺少的交易日 |
| `v_workbooks` | raw Excel 的檔案大小、修改時間與 TEJ 查詢契約 |
| `v_workbook_status` | Excel 宣告列數與處理後資料的核對、落後天數 |
| `v_ingestion_runs` | pipeline 執行紀錄、狀態、耗時與錯誤 |

另外有一個 table macro：`adj_daily_prices_between(start, end)`。它仍回傳主資料，但會利用 `year/month` 分區縮小讀取範圍，是日期查詢的首選。


In [ ]:
views = con.execute(
    """
    SELECT table_name AS view_name
    FROM information_schema.tables
    WHERE table_type = 'VIEW'
    ORDER BY table_name
    """
).df()

views["rows"] = [
    con.execute(f'SELECT count(*) FROM "{name}"').fetchone()[0]
    for name in views["view_name"]
]
views


### 主資料的 40 個欄位

不需要一開始全背。分析時通常只選需要的欄位，以下依用途分組：

| 類別 | 欄位 | 說明／單位 |
|---|---|---|
| 識別 | `date`, `stock_id`, `stock_name`, `market` | 日期、證券代碼、名稱、市場別 |
| 還原價格 | `open`, `high`, `low`, `close` | 還原開高低收，單位 NTD |
| 交易 | `volume_k_shares`, `turnover_k`, `transactions` | 成交量（千股）、成交金額（千元）、成交筆數 |
| 報酬／週轉 | `return_pct`, `return_ln`, `turnover_pct` | 簡單報酬率%、對數報酬率、週轉率% |
| 規模／權重 | `shares_outstanding_k`, `market_cap_m`, `market_cap_weight_pct`, `turnover_weight_pct` | 流通股數（千股）、市值（百萬元）與權重% |
| 估值／殖利率 | `pe_tse`, `pe_tej`, `pbr_tse`, `pbr_tej`, `psr_tej`, `div_yield_tse`, `cash_div_yield` | 本益比、股價淨值比、股價營收比、殖利率 |
| 價格變化 | `price_change`, `high_low_spread_pct` | 價格變動與高低價差% |
| 未還原價 | `bid`, `offer`, `next_ref_price`, `next_limit_up`, `next_limit_down` | 原始尺度的買賣價與次日參考／漲跌停價 |
| 狀態旗標 | `limit_flag`, `attention_flag`, `disposition_flag`, `full_delivery_flag` | 漲跌停、注意、處置、全額交割狀態 |
| 血緣／分區 | `source_file`, `ingested_at`, `year`, `month` | 來源檔、匯入時間、Parquet 分區鍵 |

資料保留 TEJ 原單位，沒有再縮放。缺值會成為 SQL 的 `NULL`，轉成 Pandas 後通常顯示為 `NaN`／`NA`。


In [ ]:
# 隨時以 DB 當下的 schema 為準：
schema = con.execute("DESCRIBE v_adj_daily_prices").df()
schema[["column_name", "column_type", "null"]]


## 4. 只學現在用得到的 SQL

SQL 的書寫順序可先記成：**從哪裡取、選哪些列、如何排序、取幾筆**。

| SQL | 意思 | Pandas 類比 |
|---|---|---|
| `SELECT a, b` | 選欄位 | `df[["a", "b"]]` |
| `FROM view` | 指定資料來源 | 使用某個 `df` |
| `WHERE condition` | 篩列 | `df.loc[condition]` |
| `ORDER BY date` | 排序 | `df.sort_values("date")` |
| `LIMIT 5` | 只取前 5 列 | `df.head(5)` |
| `GROUP BY stock_id` | 分組彙總 | `df.groupby("stock_id")` |

SQL 關鍵字大小寫沒有差別。字串與日期要放在單引號中，例如 `'2330'`、`'2024-01-01'`。


In [ ]:
query = """
SELECT date, stock_id, stock_name, open, high, low, close,
       volume_k_shares, return_pct
FROM adj_daily_prices_between(?, ?)
WHERE stock_id = ?
ORDER BY date
"""

# ? 是參數的位置；值另外傳入，可避免引號錯誤與 SQL injection。
tsmc = con.execute(
    query,
    ["2024-01-01", "2024-03-31", "2330"],
).df()

tsmc.head()


### 為什麼不先整張讀進 Pandas 再篩？

下面兩種做法結果類似，但第一種通常更快、更省記憶體：

```python
# 建議：DuckDB 先選欄、篩日期與股票，再轉 DataFrame
df = con.execute("SELECT ... WHERE ...").df()

# 避免：把 65 萬列與 40 欄全部搬進記憶體後才篩
all_data = con.execute("SELECT * FROM v_adj_daily_prices").df()
df = all_data.loc[...]
```

原則是：**SQL 做取資料與初步彙總，Pandas 做後續清理、重塑、視覺化與模型分析。**


## 5. 不想每次寫 SQL：包成 Python 函式

以下函式支援單檔或多檔證券，且使用參數化查詢。請傳入**字串**股票代碼；`'0050'` 若寫成整數 `50`，前導零會消失而查不到。


In [ ]:
def load_prices(
    stock_ids: list[str],
    start: str,
    end: str,
) -> pd.DataFrame:
    """讀取指定證券與日期範圍，回傳適合分析的常用欄位。"""
    if not stock_ids:
        raise ValueError("stock_ids 不可為空")
    if not all(isinstance(stock_id, str) for stock_id in stock_ids):
        raise TypeError("股票代碼請使用字串，例如 ['2330', '0050']")

    placeholders = ", ".join(["?"] * len(stock_ids))
    sql = f"""
        SELECT date, stock_id, stock_name, open, high, low, close,
               volume_k_shares, turnover_k, return_pct,
               market_cap_m, pe_tej, pbr_tej, cash_div_yield, market
        FROM adj_daily_prices_between(?, ?)
        WHERE stock_id IN ({placeholders})
        ORDER BY stock_id, date
    """
    return con.execute(sql, [start, end, *stock_ids]).df()


In [ ]:
prices = load_prices(
    stock_ids=["2330", "2317", "0050"],
    start="2024-01-01",
    end="2024-12-31",
)

print(prices.shape)
prices.head()


## 6. 回到熟悉的 Pandas：重塑與分析

查詢結果已是一般 `DataFrame`。例如將長表 pivot 成「日期 × 股票」的收盤價寬表，再把各序列起點標準化為 100：


In [ ]:
close_wide = (
    prices.pivot(index="date", columns="stock_id", values="close")
    .sort_index()
)
normalized_close = close_wide.div(close_wide.iloc[0]).mul(100)

normalized_close.head()


In [ ]:
# 熟悉的 Pandas 操作可以直接接在查詢結果後面。
tsmc_analysis = (
    tsmc.set_index("date")
    .assign(close_ma20=lambda df: df["close"].rolling(20).mean())
)
tsmc_analysis[["close", "close_ma20"]].tail()


## 7. 讓 DuckDB 先彙總：月資料範例

若只需要月統計，不必把每日資料全部傳給 Pandas。SQL 的 `GROUP BY` 相當於 Pandas 的 `groupby()`；以下直接回傳每月最後收盤價、月內累計成交金額與平均日報酬。


In [ ]:
monthly_sql = """
SELECT
    date_trunc('month', date) AS month,
    stock_id,
    last(close ORDER BY date) AS month_end_close,
    sum(turnover_k) AS turnover_k,
    avg(return_pct) AS avg_daily_return_pct
FROM adj_daily_prices_between(?, ?)
WHERE stock_id IN (?, ?)
GROUP BY 1, 2
ORDER BY stock_id, month
"""

monthly = con.execute(
    monthly_sql,
    ["2024-01-01", "2024-12-31", "2330", "2317"],
).df()
monthly.head()


## 8. 分析前一定要知道的資料特性

### 8.1 只有 OHLC 是還原價

`open`, `high`, `low`, `close` 是會隨除權息回溯調整的還原價；`bid`, `offer`, `next_ref_price`, `next_limit_up`, `next_limit_down` 是未還原價格。兩組尺度不同，**不可直接互相比較或相減**。例如 2330 在 2000-09-04 的還原收盤價為 27.99，但未還原的次日參考價為 133.50。

### 8.2 還原歷史值會改變

新發生的股利或分割會重寫過去的 OHLC，因此今天查到的早期 `close` 未必永遠固定。若需要可重現研究，應記錄資料版本或 `ingested_at`。

### 8.3 股票代碼不是數字

`stock_id` 是 `VARCHAR`。ETF 代碼 `'0050'` 的前導零有意義；不要先轉成 `int`。

### 8.4 缺資料不一定是 pipeline 壞掉

停牌、下市、合併或尚未上市都可能造成空白。比較多檔股票時，不要未檢查就一律 `fillna(0)`；價格的 0 通常不是合理替代值。

### 8.5 價格聯集不等於當日指數成分

367 份價格 workbook 是 365 檔歷史成分聯集加上 0050、0051 兩支 ETF。若把 365 檔全部放進每個歷史日期，會納入日後才成為成分股的公司，形成 look-ahead bias。歷史回測要用 `index_members_on('TWN50', <date>)` 或 `v_index_constituents` 依 `date` 取當日成分（見第 3 節的範例）。

每日成分數不是固定的 50／100：實測 TWN50 是 46–51 檔、TM100 是 98–102 檔，2002 年的 TWN50 只有 46–49 檔、權重合計 93.69%，這不是資料壞掉。權重也不保證加總為 100%，若要正規化必須明確記錄。

`v_index_constituents` 的 `date` 是**生效日**，內容是前一交易日的資料，所以這份資料比價格表**領先一個交易日**。

另外注意 **TWN50 指數成分 ≠ 0050 ETF 持股**：ETF 複製有落差、換股也落後生效日一兩天。若結論是關於 ETF 而不是指數，必須說明用的是指數成分作為代理。


## 9. 查詢資料新鮮度與品質

先看健康摘要。`gaps` 是個別證券在自身活躍期間缺的交易日數；目前已知的少量缺口可能來自真實公司事件。`noncurrent_securities` 也會包含已下市或合併的證券，不可單憑此數字判定更新失敗。


In [ ]:
health_sql = """
SELECT
    (SELECT date FROM v_last_trading_day) AS latest_day,
    (SELECT count(*) FROM v_securities) AS securities,
    (SELECT count(*) FROM v_security_gaps) AS gaps,
    (SELECT count(*) FROM v_thin_trading_days) AS thin_days,
    (SELECT count(*) FROM v_securities WHERE NOT is_current)
        AS noncurrent_securities
"""
con.execute(health_sql).df()


In [ ]:
# 最落後的 workbook；rows_delta != 0 才表示 Excel 宣告列數與解析列數不一致。
workbook_status = con.execute(
    """
    SELECT stock_id, stock_name, last_date, days_behind, rows_delta,
           query_matches_contract
    FROM v_workbook_status
    ORDER BY days_behind DESC, stock_id
    LIMIT 20
    """
).df()
workbook_status


In [ ]:
# 最近的 pipeline 執行紀錄
recent_runs = con.execute(
    """
    SELECT job, status, started_at, duration_seconds, error_count
    FROM v_ingestion_runs
    ORDER BY started_at DESC
    LIMIT 10
    """
).df()
recent_runs


## 10. 常用查詢速查

```sql
-- 有哪些證券？
SELECT * FROM v_securities ORDER BY stock_id;

-- 某檔證券最近 10 筆（結果改成由舊到新）
SELECT * FROM (
    SELECT date, stock_id, close, return_pct
    FROM v_adj_daily_prices
    WHERE stock_id = '2330'
    ORDER BY date DESC
    LIMIT 10
) ORDER BY date;

-- 指定日期與多檔股票（日期範圍請用 macro）
SELECT date, stock_id, close
FROM adj_daily_prices_between('2024-01-01', '2024-03-31')
WHERE stock_id IN ('2330', '2317', '0050')
ORDER BY stock_id, date;

-- 檢查某些欄位是否缺值
SELECT
    count(*) AS rows,
    count(*) FILTER (WHERE pe_tej IS NULL) AS missing_pe,
    count(*) FILTER (WHERE pbr_tej IS NULL) AS missing_pbr
FROM adj_daily_prices_between('2024-01-01', '2024-12-31');
```

把任一段 SQL 放進 `con.execute(sql).df()`，就會得到 Pandas DataFrame。實務上建議像前面範例一樣使用 `?` 傳參數，而不是用 f-string 拼接使用者輸入。


## 11. 收尾與下一步

最常見的工作流程是：

1. 用 `v_securities` 確認研究對象是否存在與涵蓋日期。
2. 用 `adj_daily_prices_between()` 限定日期，並在 SQL 中只選需要的欄位。
3. 用 `.df()` 取得 DataFrame，再進行熟悉的 Pandas 分析。
4. 發表結果前查看 `v_last_trading_day`、`v_security_gaps` 與 `v_ingestion_runs`。

分析結束可關閉連線；若忘了關，kernel 結束時也會釋放。


In [ ]:
con.close()
print("DuckDB 連線已關閉")
